In [ ]:
!pip install -U -q "google-genai"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 724.7/724.7 kB 18.2 MB/s eta 0:00:00


In [ ]:
from google.cloud import storage
from typing import List, Dict, Tuple
from tqdm import tqdm
import time
import datetime
import json
import os

# basic stuff
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, recall_score, confusion_matrix

# these used for gemini client initialization specifically
from google import genai
from google.genai import types
from google.colab import userdata

In [ ]:
# GCS authentication w/ service account
service_account_info_str = userdata.get('GCS_SERVICE_ACCOUNT_KEY')
temp_key_file_path = "/tmp/gcs_service_account_key.json"
with open(temp_key_file_path, "w") as f:
    f.write(service_account_info_str)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = temp_key_file_path

In [ ]:
from google.colab import userdata
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_TOKEN')
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: christopher-w-song (christopher-w-song-dougherty-valley-high-school) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### main classification function

In [ ]:
def classify_sample(client, model_name: str, classification_prompt: str,
                    tkdname: str, classification_type: str, collect_reasoning: bool,
                    transcription: str = None) -> dict:
    """
    Classify a single sample (text or audio) using Gemini

    Args:
        client: Google Gen AI client
        model_name: Name of Gemini model to use (e.g., 'gemini-2.5-flash-latest')
        classification_prompt: The prompt to guide the model's classification
        tkdname: The tkdname identifier
        classification_type: "text", "audio", or "multimodal"
        collect_reasoning: Whether to collect reasoning from the model
        transcription: Text transcription (required for text and multimodal classification)

    Returns:
        Dictionary with classification results
    """

    contents = [classification_prompt]

    # Add audio part if needed (audio or multimodal)
    if classification_type in ["audio", "multimodal"]:
        audio_blob_name = f"flattened-audio-data/{tkdname}"
        gcs_uri = f"gs://{bucket_name}/{audio_blob_name}"

        audio_part = types.Part(
            file_data=types.FileData(
                file_uri=gcs_uri,
                mime_type="audio/wav"
            )
        )
        contents.append(audio_part)

    # Add transcription if needed (text or multimodal)
    if classification_type in ["text", "multimodal"]:
        contents.append(transcription)


    start_time = time.time()

    config_dict = {
        "temperature": 1,
        "response_mime_type": "text/plain"
    }
    if collect_reasoning:
        config_dict["thinking_config"] = types.ThinkingConfig(include_thoughts=True)

    # generate response with selected contents
    response = client.models.generate_content(
        model = model_name,
        contents = contents,
        config = types.GenerateContentConfig(**config_dict)
    )

    end_time = time.time()
    response_time = round((end_time - start_time), 2)

    # extract response, reasoning, & finish reason
    reasoning_text = None
    response_text = None
    if response.candidates and response.candidates[0].content.parts:
        for part in response.candidates[0].content.parts:
            if not part.text:
                continue

            if collect_reasoning and hasattr(part, 'thought') and part.thought:
                reasoning_text = part.text
            else:
                response_text = part.text.strip().upper()  # Ensure consistent format

    finish_reason = response.candidates[0].finish_reason

    # extract metadata
    usage_metadata = response.usage_metadata
    input_tokens = usage_metadata.prompt_token_count
    output_tokens = usage_metadata.candidates_token_count
    thoughts_tokens = usage_metadata.thoughts_token_count
    total_tokens = usage_metadata.total_token_count

    result = {
        'prediction': response_text,
        'response_time': response_time,
        'finish_reason': finish_reason,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'thoughts_tokens': thoughts_tokens,
        'total_tokens': total_tokens,
    }

    if collect_reasoning:
        result['reasoning_text'] = reasoning_text

    return result

### helper functions

In [ ]:
# helper functions

def save_results(results_df: pd.DataFrame, run_name: str, reasoning_dict: dict = None) -> str:
    """
    Save results to wandb artifacts

    Args:
        results_df: DataFrame with classification results.
        run_name: Base name for the run.
        reasoning_dict: Dictionary with reasoning text.

    Returns:
        String with the results CSV filename that was used.
    """

    # Log results as W&B artifact
    csv_filename = f"results_{run_name}.csv"
    csv_path = f"/content/{csv_filename}"
    results_df.to_csv(csv_path, index=False)

    #csv_path = os.path.join(os.getcwd(), csv_filename)
    #json_path = os.path.join(os.getcwd(), f"reasoning_{run_name}.json")

    results_artifact = wandb.Artifact(f"results_{run_name}", type="results")
    results_artifact.add_file(csv_path)
    wandb.log_artifact(results_artifact)
    print(f"✓ Results logged to W&B")

    # Log reasoning as W&B artifact
    if reasoning_dict is not None:
        json_path = f"/content/reasoning_{run_name}.json"
        with open(json_path, 'w') as f:
            json.dump(reasoning_dict, f, indent=2)
        reasoning_artifact = wandb.Artifact(f"reasoning_{run_name}", type="reasoning")
        reasoning_artifact.add_file(json_path)
        wandb.log_artifact(reasoning_artifact)
        print(f"✓ Reasoning logged to W&B")

    return csv_filename

def evaluate_results(results_df: pd.DataFrame, run_name: str) -> dict:
    """
    Evaluate classification results from a DataFrame

    Args:
        results_df: DataFrame containing the classification results.
        run_name: Base name for the run.

    Returns:
        Dictionary with evaluation metrics.
    """

    from sklearn.metrics import accuracy_score, f1_score, recall_score

    def calculate_subset_metrics(subset_df, subset_name):
        # calculate acc, UAR, micro f1, macro F1 for one subset

        # filter out errors
        valid_df = subset_df[subset_df['prediction'] != "ERROR"]

        y_true = valid_df['dx'].values
        y_pred = valid_df['prediction'].values

        return {
            "subset": subset_name,
            "samples": len(subset_df),
            "valid_samples": len(valid_df),
            "accuracy": round(accuracy_score(y_true,y_pred), 4),
            "uar": round(recall_score(y_true, y_pred, average='macro', zero_division=0), 4),
            "micro_f1": round(f1_score(y_true, y_pred, average='micro', zero_division=0), 4),
            "macro_f1": round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4)
        }

    all_data = calculate_subset_metrics(results_df, "all_data")
    english_data = results_df[results_df['language'].astype(str).str.lower().isin(['en', 'english'])]
    english_metrics = calculate_subset_metrics(english_data, "english_only")
    chinese_data = results_df[results_df['language'].astype(str).str.lower().isin(['zh', 'chinese', 'cn'])]
    chinese_metrics = calculate_subset_metrics(chinese_data, "chinese_only")

    print(f"\n📊 EVALUATION RESULTS:")
    print(f"{'Subset':<15} {'Valid/Total':<12} {'Accuracy':<10} {'UAR':<10} {'Micro F1':<10} {'Macro F1':<10}")
    print("-" * 75)
    for metrics in [all_data, english_metrics, chinese_metrics]:
        sample_str = f"{metrics['valid_samples']}/{metrics['samples']}"
        print(f"{metrics['subset']:<15} {sample_str:<12} {metrics['accuracy']:<10} {metrics['uar']:<10} {metrics['micro_f1']:<10} {metrics['macro_f1']:<10}")

    # Log to W&B
    wandb.run.summary.update({
        # All data
        "accuracy_all": all_data["accuracy"],
        "uar_all": all_data["uar"],
        "micro_f1_all": all_data["micro_f1"],
        "macro_f1_all": all_data["macro_f1"],
        "valid_samples_all": all_data["valid_samples"],

        # English only
        "accuracy_english": english_metrics["accuracy"],
        "uar_english": english_metrics["uar"],
        "micro_f1_english": english_metrics["micro_f1"],
        "macro_f1_english": english_metrics["macro_f1"],
        "valid_samples_english": english_metrics["valid_samples"],

        # Chinese only
        "accuracy_chinese": chinese_metrics["accuracy"],
        "uar_chinese": chinese_metrics["uar"],
        "micro_f1_chinese": chinese_metrics["micro_f1"],
        "macro_f1_chinese": chinese_metrics["macro_f1"],
        "valid_samples_chinese": chinese_metrics["valid_samples"]
    })

    print("✓ Metrics logged to W&B")

    return {
        "all_data": all_data,
        "english_only": english_metrics,
        "chinese_only": chinese_metrics
    }

### driver code

In [ ]:
# args
bucket_name = "taukadial-25"
groundtruth_path = "groundtruth/groundtruth_combined_lang2.csv"
transcription_path = "text-data/transcription_data_modified.csv"
model_name = "gemini-2.5-pro"
collect_reasoning = False
temperature = 1
classification_prompt = """
Assess the cognitive condition based on the input audio and text data, where an elderly speaker describes one of three images as part of a clinician-guided task.
Indicate the diagnosis using one of these labels: NC (Normal Cognitive) or MCI (Mild Cognitive Impairment).
Output only "NC" or "MCI" as your response. Do not include any explanation, reasoning, or additional text.
"""
classification_type = "audio" # audio, text, or multimodal, make sure to change the prompt to reflect this
timestamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
run_name = f"zero-shot_{classification_type}_{model_name}_{timestamp}"

# GCS initialization
storage_client = storage.Client()
gcs_bucket = storage_client.bucket(bucket_name)

# Gemini Client initialization
client = genai.Client(
    #api_key = userdata.get('GOOGLE_API_KEY'),
    vertexai = True,
    project = "grand-store-465802-r4",
    location = "us-west1"
)

# wandb initialization
run = wandb.init(
    project = "zero-shot-mci-classification",
    name = run_name,
    config = {
        "model_name": model_name,
        "model_type": "closed_source",
        "model_family": "gemini",
        "bucket_name": bucket_name,
        "groundtruth_path": groundtruth_path,
        #"samples_processed": 507,  # Update with actual size
        "temperature": temperature,
        "prompt": classification_prompt
    }
)

from io import StringIO
blob = gcs_bucket.blob(groundtruth_path)
content = blob.download_as_text()
results_df = pd.read_csv(StringIO(content))

# load transcription data & create lookup map, tehcnically only needed for text & multimodal
transcription_blob = gcs_bucket.blob(transcription_path)
transcription_content = transcription_blob.download_as_text()
transcription_df = pd.read_csv(StringIO(transcription_content))
transcription_map = transcription_df.set_index('tkdname')['transcription'].to_dict()

results_df['prediction'] = None
results_df['response_time'] = None
results_df['finish_reason'] = None
results_df['input_tokens'] = None
results_df['output_tokens'] = None
results_df['thoughts_tokens'] = None
results_df['total_tokens'] = None

reasoning_dict = {} if collect_reasoning else None

# Iterate through each row and process corresponding data
for idx, row in tqdm(results_df.iterrows(), total=len(results_df), desc=f"Processing {classification_type} transcriptions: ", colour='green'):
    tkdname = row['tkdname']

    # get transcription if text classification
    transcription = transcription_map.get(tkdname) if classification_type in ["text", "multimodal"] else None

    #generalized
    classification_result = classify_sample(
        client, model_name, classification_prompt,
        tkdname, classification_type, collect_reasoning, transcription)

    if classification_result and isinstance(classification_result, dict):
        results_df.at[idx, 'prediction'] = classification_result['prediction']
        results_df.at[idx, 'response_time'] = classification_result['response_time']
        results_df.at[idx, 'finish_reason'] = classification_result['finish_reason']

        results_df.at[idx, 'input_tokens'] = classification_result['input_tokens']
        results_df.at[idx, 'output_tokens'] = classification_result['output_tokens']
        results_df.at[idx, 'thoughts_tokens'] = classification_result['thoughts_tokens']
        results_df.at[idx, 'total_tokens'] = classification_result['total_tokens']

        if collect_reasoning: reasoning_dict[tkdname] = classification_result.get('reasoning_text', None)

        # sample level logging, only numerical metrics
        wandb.log({
            "response_time": classification_result['response_time'],
            "input_tokens": classification_result['input_tokens'],
            "output_tokens": classification_result['output_tokens'],
            "thoughts_tokens": classification_result['thoughts_tokens'],
            "total_tokens": classification_result['total_tokens']
        })

    else:
        results_df.at[idx, 'prediction'] = 'ERROR'
        if collect_reasoning: reasoning_dict[tkdname] = None

    time.sleep(1) # Rate limiting

save_results(results_df, run_name, reasoning_dict)
evaluate_results(results_df, run_name)

print(f"\nSummary:")
print(f"  Total samples processed: {len(results_df)}")
print(f"  Valid predictions: {len(results_df[results_df['prediction'] != 'ERROR'])}")
print(f"  Error cases: {len(results_df[results_df['prediction'] == 'ERROR'])}")
print(f"  Skipped (no text): {len(results_df[results_df['prediction'] == 'SKIPPED_NO_TEXT'])}")

# wandb log final summary metrics
total_samples = len(results_df)
total_tokens = results_df['total_tokens'].sum()
total_runtime = results_df['response_time'].sum()
wandb.run.summary.update({
    "total_runtime_minutes": total_runtime / 60,
    "total_tokens_used": total_tokens,
    "avg_tokens_per_sample": total_tokens / total_samples,
    "samples_per_minute": total_samples / (total_runtime / 60)
})
wandb.finish()
print("✓ W&B run completed and logged")

input_tokens,█▃▄▁▂▂▁▂
output_tokens,▁▁▁█████
response_time,▁▁▁▁▁▁█▁
thoughts_tokens,▁▁▁▁▁▁█▁
total_tokens,▁▁▁▁▁▁█▁
input_tokens,1035
output_tokens,2
response_time,16.6
thoughts_tokens,1688
total_tokens,2725


Processing audio transcriptions: 100%|██████████| 507/507 [2:03:27<00:00, 14.61s/it]


✓ Results logged to W&B

📊 EVALUATION RESULTS:
Subset          Valid/Total  Accuracy   UAR        Micro F1   Macro F1  
---------------------------------------------------------------------------
all_data        507/507      0.5424     0.5402     0.5424     0.5391    
english_only    246/246      0.5244     0.5881     0.5244     0.5177    
chinese_only    261/261      0.5594     0.5563     0.5594     0.5231    
✓ Metrics logged to W&B

Summary:
  Total samples processed: 507
  Valid predictions: 507
  Error cases: 0
  Skipped (no text): 0


input_tokens,▄▂▂▄▂▂▂▃▂▃▃▂▁▅▄▁▂▂▂▁▂▂▂▁▁▂▂▂▂▃▂▂▁▂▃▂▆▂█▄
output_tokens,█▁▁▁████▁▁███▁▁▁▁▁███▁▁▁▁█▁█▁███▁▁▁█▁▁█▁
response_time,▂▂▃▁▇▄▃▃▆▄▄▁█▄▅▁▆▆▅▄▅▂▅█▄▃▅▅▅▂▆▇█▂▃▃▅▄▄▃
thoughts_tokens,▂▃▃▆█▆▄▇▄█▂▁▆▃▃▃▁▇▄▂▇▄▅▅▃▂▁▁▅▄▂▄▅▅▅▄▃▁▄▃
total_tokens,▁▂▃▃▂▃▂▃▁█▂▂▃▄▂▁▃▄▂▂▂▃▂▅▁▂▃▂▂▄▁▂▂▁▂▅▃▂▂▃
accuracy_all,0.5424
accuracy_chinese,0.5594
accuracy_english,0.5244
avg_tokens_per_sample,2820.1854
input_tokens,1635
macro_f1_all,0.5391


✓ W&B run completed and logged


### test & extra

In [ ]:
def classify_text(client, model_name: str, classification_prompt: str, text_data: str, collect_reasoning:bool) -> dict:
    """
    Classify a single piece of text data using Gemini

    Args:
        client: Google Gen AI client
        model_name: Name of Gemini model to use (e.g., 'gemini-1.5-flash-latest')
        classification_prompt: The prompt to guide the model's classification.
        tkdname: filename for the sample being classified
        collect_reasoning: Whether to collect reasoning from the model.

    Returns:
        Dictionary with classification results
    """

    start_time = time.time()

    config_dict = {
        "temperature": 1,
        "response_mime_type": "text/plain"
    }
    if collect_reasoning:
        config_dict["thinking_config"] = types.ThinkingConfig(include_thoughts=True)

    # Pass text data directly in contents
    response = client.models.generate_content(
        model=model_name,
        contents=[classification_prompt, text_data],
        config=types.GenerateContentConfig(**config_dict)
    )

    end_time = time.time()
    response_time = round((end_time - start_time), 2)

    usage_metadata = response.usage_metadata if hasattr(response, 'usage_metadata') else None

    reasoning_text = None
    response_text = None
    if response.candidates and response.candidates[0].content.parts:
        for part in response.candidates[0].content.parts:
            if not part.text:
                continue

            if collect_reasoning and hasattr(part, 'thought') and part.thought:
                reasoning_text = part.text
            else:
                response_text = part.text.strip().upper() # Ensure consistent format

    finish_reason = response.candidates[0].finish_reason

    input_tokens = usage_metadata.prompt_token_count
    output_tokens = usage_metadata.candidates_token_count
    thoughts_tokens = usage_metadata.thoughts_token_count
    total_tokens = usage_metadata.total_token_count

    result = {
        'prediction': response_text,
        'response_time': response_time,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'thoughts_tokens': thoughts_tokens,
        'total_tokens': total_tokens,
        'finish_reason': finish_reason,
    }

    if collect_reasoning:
        result['reasoning_text'] = reasoning_text

    return result

def classify_audio(client, model_name: str, classification_prompt: str, bucket_name: str, tkdname: str, collect_reasoning: bool) -> dict:
    """
    Classify a single audio file using Gemini

    Args:
        client: Google Gen AI client
        model_name: Name of Gemini model to use (e.g., 'gemini-2.5-flash-latest')
        classification_prompt: The prompt to guide the model's classification
        bucket_name: Name of the GCS bucket containing audio files
        tkdname: The tkdname identifier to construct audio file path
        collect_reasoning: Whether to collect reasoning from the model

    Returns:
        Dictionary with classification results
    """

    # Construct audio file path using naming convention
    audio_blob_name = f"audio/{tkdname}.wav"  # Adjust path as needed
    gcs_uri = f"gs://{bucket_name}/{audio_blob_name}"

    # Create Part from GCS URI
    audio_part = types.Part(
        file_data=types.FileData(
            file_uri=gcs_uri,
            mime_type="audio/wav"
        )
    )

    start_time = time.time()

    config_dict = {
        "temperature": 1,
        "response_mime_type": "text/plain"
    }
    if collect_reasoning:
        config_dict["thinking_config"] = types.ThinkingConfig(include_thoughts=True)

    # Pass audio part in contents
    response = client.models.generate_content(
        model=model_name,
        contents=[classification_prompt, audio_part],
        config=types.GenerateContentConfig(**config_dict)
    )

    end_time = time.time()
    response_time = round((end_time - start_time), 2)

    usage_metadata = response.usage_metadata if hasattr(response, 'usage_metadata') else None

    reasoning_text = None
    response_text = None
    if response.candidates and response.candidates[0].content.parts:
        for part in response.candidates[0].content.parts:
            if not part.text:
                continue

            if collect_reasoning and hasattr(part, 'thought') and part.thought:
                reasoning_text = part.text
            else:
                response_text = part.text.strip().upper()  # Ensure consistent format

    finish_reason = response.candidates[0].finish_reason

    input_tokens = usage_metadata.prompt_token_count
    output_tokens = usage_metadata.candidates_token_count
    thoughts_tokens = usage_metadata.thoughts_token_count
    total_tokens = usage_metadata.total_token_count

    result = {
        'prediction': response_text,
        'response_time': response_time,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'thoughts_tokens': thoughts_tokens,
        'total_tokens': total_tokens,
        'finish_reason': finish_reason,
    }

    if collect_reasoning:
        result['reasoning_text'] = reasoning_text

    return result

In [ ]:
# Test single multimodal classification

import time
import datetime
from google.cloud import storage
import pandas as pd
from io import StringIO

from google import genai
from google.genai import types
from google.colab import userdata
import os

# Setup
bucket_name = "taukadial-25"
model_name = "gemini-2.5-pro"
transcription_path = "text-data/transcription_data_modified.csv"
collect_reasoning = False
classification_prompt = """
Assess the cognitive condition based on the provided audio and text data, where an elderly speaker describes one of three images as part of a clinician-guided task.
Indicate the diagnosis using one of these labels: NC (Normal Cognitive) or MCI (Mild Cognitive Impairment).
Output only NC or MCI as your response. Do not include any explanation, reasoning, or additional text.
"""

# Initialize Gemini client
client = genai.Client(
    vertexai=True,
    project="grand-store-465802-r4",
    location="us-west1"
)

# Initialize GCS and load transcription data
storage_client = storage.Client()
gcs_bucket = storage_client.bucket(bucket_name)
transcription_blob = gcs_bucket.blob(transcription_path)
transcription_content = transcription_blob.download_as_text()
transcription_df = pd.read_csv(StringIO(transcription_content))
transcription_map = transcription_df.set_index('tkdname')['transcription'].to_dict()

# Test with first available sample
test_tkdname = list(transcription_map.keys())[0]
test_transcription = transcription_map[test_tkdname]

print(f"Testing multimodal classification with sample: {test_tkdname}")
print(f"Transcription length: {len(test_transcription)} characters")
print(f"Transcription preview: {test_transcription[:100]}...")

# Check if audio file exists
audio_blob_name = f"audio-data/{test_tkdname}"
audio_blob = gcs_bucket.blob(audio_blob_name)
print(f"Audio file: {audio_blob_name}")
print(f"Audio file exists: {audio_blob.exists()}")
if audio_blob.exists():
    print(f"Audio file size: {audio_blob.size} bytes")

# Run multimodal classification
test_result = classify_sample(
    client,
    model_name,
    classification_prompt,
    test_tkdname,
    "multimodal",
    collect_reasoning,
    test_transcription
)

print(f"\nResults:")
print(f"Sample: {test_tkdname}")
print(f"Prediction: {test_result['prediction']}")
print(f"Response time: {test_result['response_time']}s")
print(f"Total tokens: {test_result['total_tokens']}")
print(f"Input tokens: {test_result['input_tokens']}")
print(f"Output tokens: {test_result['output_tokens']}")
print(f"Finish reason: {test_result['finish_reason']}")

if collect_reasoning and 'reasoning_text' in test_result:
    print(f"Reasoning: {test_result['reasoning_text']}")

Testing multimodal classification with sample: taukdial-002-1.wav
Transcription length: 1466 characters
Transcription preview: Yes. Do you need to zoom in or anything? No. OK, so I'm going to have you tell me a story with a beg...
Audio file: audio-data/taukdial-002-1.wav
Audio file exists: True
Audio file size: None bytes

Results:
Sample: taukdial-002-1.wav
Prediction: NC
Response time: 15.15s
Total tokens: 4711
Input tokens: 3485
Output tokens: 1
Finish reason: FinishReason.STOP
